# 🔥 NAFNet-SR: Thermal Image 4× Super-Resolution
**Google Colab (Free Tier — T4 GPU)**

Pipeline: LR thermal (90×60) → NAFNet-SR → HR thermal (360×240)

---

## 1. GPU Check & Setup

In [ ]:
import torch
import os

# Check GPU
assert torch.cuda.is_available(), "❌ No GPU! Go to Runtime → Change runtime type → T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}")

## 2. Mount Google Drive
Mount Drive to save checkpoints & logs (persists across runtime restarts).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folder on Drive
DRIVE_PROJECT = '/content/drive/MyDrive/thermal_sr'
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/results', exist_ok=True)
print(f"✅ Project folder: {DRIVE_PROJECT}")

## 3. Dataset Setup

**Edit the paths below** to point to your LR and HR image folders.

Expected structure:
```
your_dataset/
├── train/
│   ├── lr/    ← Low-res thermal (90×60)
│   └── hr/    ← High-res thermal (360×240)
└── val/
    ├── lr/
    └── hr/
```
Filenames in `lr/` and `hr/` must match (sorted order).

In [ ]:
#@title ⚙️ **EDIT THESE PATHS** { display-mode: "form" }

# ========== DATASET PATHS (EDIT THIS!) ==========
TRAIN_LR_DIR = '/content/drive/MyDrive/your_dataset/train/lr'   #@param {type:"string"}
TRAIN_HR_DIR = '/content/drive/MyDrive/your_dataset/train/hr'   #@param {type:"string"}
VAL_LR_DIR   = '/content/drive/MyDrive/your_dataset/val/lr'     #@param {type:"string"}
VAL_HR_DIR   = '/content/drive/MyDrive/your_dataset/val/hr'     #@param {type:"string"}

# ========== TRAINING SETTINGS ==========
EPOCHS        = 200    #@param {type:"integer"}
BATCH_SIZE    = 8      #@param {type:"integer"}
LEARNING_RATE = 1e-3   #@param {type:"number"}
PATCH_SIZE    = 128    #@param {type:"integer"}
UPSCALE       = 4      #@param {type:"integer"}
NUM_BLOCKS    = 16     #@param {type:"integer"}
WIDTH         = 48     #@param {type:"integer"}

# ========== SAVE LOCATIONS (on Drive) ==========
SAVE_DIR = f'{DRIVE_PROJECT}/checkpoints'
LOG_DIR  = f'{DRIVE_PROJECT}/logs'

# Verify dataset exists
for d, name in [(TRAIN_LR_DIR, 'Train LR'), (TRAIN_HR_DIR, 'Train HR'),
                (VAL_LR_DIR, 'Val LR'), (VAL_HR_DIR, 'Val HR')]:
    if os.path.exists(d):
        count = len([f for f in os.listdir(d) if f.lower().endswith(('.png','.jpg','.jpeg','.tif','.bmp'))])
        print(f"✅ {name}: {d} ({count} images)")
    else:
        print(f"❌ {name}: {d} NOT FOUND — fix the path!")

## 4. Model & Training Code
All code is embedded below — nothing to upload.

In [ ]:
#@title 4a. NAFNet-SR Model Architecture (run this cell)

import torch
import torch.nn as nn
import torch.nn.functional as F


class LayerNorm2d(nn.Module):
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        var = x.var(dim=1, keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var + self.eps)
        return x * self.weight[None,:,None,None] + self.bias[None,:,None,None]


class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class SimplifiedChannelAttention(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(num_channels, num_channels, 1, 1, 0)

    def forward(self, x):
        return x * self.conv(self.pool(x))


class NAFBlock(nn.Module):
    def __init__(self, c, dw_expand=2, ffn_expand=2):
        super().__init__()
        dw_c = c * dw_expand
        ffn_c = c * ffn_expand

        self.norm1 = LayerNorm2d(c)
        self.conv1 = nn.Conv2d(c, dw_c, 1, 1, 0)
        self.conv2 = nn.Conv2d(dw_c, dw_c, 3, 1, 1, groups=dw_c)
        self.gate1 = SimpleGate()
        self.sca = SimplifiedChannelAttention(dw_c // 2)
        self.conv3 = nn.Conv2d(dw_c // 2, c, 1, 1, 0)

        self.norm2 = LayerNorm2d(c)
        self.conv4 = nn.Conv2d(c, ffn_c, 1, 1, 0)
        self.gate2 = SimpleGate()
        self.conv5 = nn.Conv2d(ffn_c // 2, c, 1, 1, 0)

        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, x):
        y = self.norm1(x)
        y = self.conv1(y)
        y = self.conv2(y)
        y = self.gate1(y)
        y = self.sca(y)
        y = self.conv3(y)
        x = x + y * self.beta

        y = self.norm2(x)
        y = self.conv4(y)
        y = self.gate2(y)
        y = self.conv5(y)
        x = x + y * self.gamma
        return x


class PixelShuffleUpsampler(nn.Module):
    def __init__(self, channels, upscale=4):
        super().__init__()
        layers = []
        num_stages = {2: 1, 4: 2, 8: 3}[upscale]
        for _ in range(num_stages):
            layers.extend([
                nn.Conv2d(channels, channels * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.GELU(),
            ])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class NAFNetSR(nn.Module):
    def __init__(self, img_channels=3, width=48, num_blocks=16, upscale=4):
        super().__init__()
        self.upscale = upscale
        self.intro = nn.Conv2d(img_channels, width, 3, 1, 1)
        self.body = nn.Sequential(*[NAFBlock(width) for _ in range(num_blocks)])
        self.body_tail = nn.Conv2d(width, width, 3, 1, 1)
        self.upsampler = PixelShuffleUpsampler(width, upscale)
        self.ending = nn.Conv2d(width, img_channels, 3, 1, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, LayerNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        shallow = self.intro(x)
        deep = self.body(shallow)
        deep = self.body_tail(deep) + shallow
        up = self.upsampler(deep)
        return self.ending(up)


# --- Verify ---
model = NAFNetSR(img_channels=3, width=WIDTH, num_blocks=NUM_BLOCKS, upscale=UPSCALE)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ NAFNet-SR built: {total_params:.2f}M parameters")

with torch.no_grad():
    dummy = torch.randn(1, 3, 60, 90)
    out = model(dummy)
    print(f"   Shape check: {dummy.shape} → {out.shape} ({UPSCALE}× upscale)")
    assert out.shape == (1, 3, 240, 360)
    print("   ✅ Forward pass OK")
del model, dummy, out

In [ ]:
#@title 4b. Dataset & DataLoader (run this cell)

import random
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

IMG_EXT = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}


def get_image_paths(directory):
    directory = Path(directory)
    paths = sorted([p for p in directory.iterdir() if p.suffix.lower() in IMG_EXT])
    assert len(paths) > 0, f"No images found in {directory}"
    return paths


def load_image(path, channels=3):
    img = Image.open(path)
    img = img.convert('RGB') if channels == 3 else img.convert('L')
    img = np.array(img, dtype=np.float32)
    if img.max() > 255.0:
        img = img / 65535.0
    elif img.max() > 1.0:
        img = img / 255.0
    return img


class PairedSRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, patch_size=128, upscale=4, channels=3,
                 augment=True, is_train=True):
        self.lr_paths = get_image_paths(lr_dir)
        self.hr_paths = get_image_paths(hr_dir)
        assert len(self.lr_paths) == len(self.hr_paths), \
            f"Mismatch: {len(self.lr_paths)} LR vs {len(self.hr_paths)} HR"
        self.patch_size = patch_size
        self.lr_patch = patch_size // upscale
        self.upscale = upscale
        self.channels = channels
        self.augment = augment and is_train
        self.is_train = is_train
        split = 'Train' if is_train else 'Val'
        print(f"  {split}: {len(self)} pairs | LR patch: {self.lr_patch}×{self.lr_patch} → HR patch: {patch_size}×{patch_size}")

    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self, idx):
        lr = load_image(self.lr_paths[idx], self.channels)
        hr = load_image(self.hr_paths[idx], self.channels)
        if lr.ndim == 2: lr = lr[:,:,np.newaxis]
        if hr.ndim == 2: hr = hr[:,:,np.newaxis]

        if self.is_train:
            # Random crop at corresponding positions
            lr_h, lr_w = lr.shape[:2]
            lp = self.lr_patch
            top_lr = random.randint(0, max(0, lr_h - lp))
            left_lr = random.randint(0, max(0, lr_w - lp))
            lr = lr[top_lr:top_lr+lp, left_lr:left_lr+lp]
            top_hr = top_lr * self.upscale
            left_hr = left_lr * self.upscale
            hp = self.patch_size
            hr = hr[top_hr:top_hr+hp, left_hr:left_hr+hp]

        if self.augment:
            if random.random() > 0.5:
                lr = np.flip(lr, 1).copy()
                hr = np.flip(hr, 1).copy()
            if random.random() > 0.5:
                lr = np.flip(lr, 0).copy()
                hr = np.flip(hr, 0).copy()
            k = random.randint(0, 3)
            if k > 0:
                lr = np.rot90(lr, k).copy()
                hr = np.rot90(hr, k).copy()

        lr_t = torch.from_numpy(lr.transpose(2,0,1)).float().clamp(0,1)
        hr_t = torch.from_numpy(hr.transpose(2,0,1)).float().clamp(0,1)
        return {'input': lr_t, 'target': hr_t, 'filename': self.lr_paths[idx].name}


# Build dataloaders
print("Loading dataset...")
train_ds = PairedSRDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, PATCH_SIZE, UPSCALE, augment=True, is_train=True)
val_ds   = PairedSRDataset(VAL_LR_DIR, VAL_HR_DIR, PATCH_SIZE, UPSCALE, augment=False, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

# Quick sanity check — visualize one pair
sample = train_ds[0]
print(f"\n  Sample LR: {sample['input'].shape}  (min={sample['input'].min():.3f}, max={sample['input'].max():.3f})")
print(f"  Sample HR: {sample['target'].shape} (min={sample['target'].min():.3f}, max={sample['target'].max():.3f})")
print("✅ Dataset ready")

In [ ]:
#@title 4c. Loss & Metrics (run this cell)

import math

class CharbonnierLoss(nn.Module):
    """Smooth L1 — better than L1 for image restoration."""
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps_sq = eps ** 2

    def forward(self, pred, target):
        diff = pred - target
        return torch.sqrt(diff * diff + self.eps_sq).mean()


@torch.no_grad()
def calc_psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target).item()
    if mse < 1e-10: return 100.0
    return 10.0 * math.log10(max_val ** 2 / mse)


@torch.no_grad()
def calc_ssim(pred, target, window_size=11, max_val=1.0):
    C1 = (0.01 * max_val) ** 2
    C2 = (0.03 * max_val) ** 2
    sigma = 1.5
    coords = torch.arange(window_size, dtype=torch.float32, device=pred.device)
    coords -= window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window = g.unsqueeze(0) * g.unsqueeze(1)
    window = window.unsqueeze(0).unsqueeze(0)
    ch = pred.shape[1]
    window = window.expand(ch, -1, -1, -1)
    pad = window_size // 2
    mu1 = F.conv2d(pred, window, padding=pad, groups=ch)
    mu2 = F.conv2d(target, window, padding=pad, groups=ch)
    mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1*mu2
    s1_sq = F.conv2d(pred*pred, window, padding=pad, groups=ch) - mu1_sq
    s2_sq = F.conv2d(target*target, window, padding=pad, groups=ch) - mu2_sq
    s12 = F.conv2d(pred*target, window, padding=pad, groups=ch) - mu1_mu2
    ssim_map = ((2*mu1_mu2+C1)*(2*s12+C2)) / ((mu1_sq+mu2_sq+C1)*(s1_sq+s2_sq+C2))
    return ssim_map.mean().item()


print("✅ Loss & metrics ready")

## 5. Train! 🚀
Run the cell below to start training. Progress is printed live.

In [ ]:
import time
import json
from torch.cuda.amp import GradScaler, autocast

# ============ SETUP ============
device = torch.device('cuda')
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Model
model = NAFNetSR(img_channels=3, width=WIDTH, num_blocks=NUM_BLOCKS, upscale=UPSCALE).to(device)
print(f"Model: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

# Loss
criterion = CharbonnierLoss(eps=1e-3).to(device)

# Optimizer — β2=0.9 is critical for NAFNet (not 0.999!)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                               betas=(0.9, 0.9), weight_decay=0.0)

# Scheduler — cosine with 5-epoch linear warmup
total_steps = EPOCHS * len(train_loader)
warmup_steps = 5 * len(train_loader)
min_lr = 1e-6

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(min_lr / LEARNING_RATE, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed precision
scaler = GradScaler()

# Logging
log_file = f"{LOG_DIR}/train_log.jsonl"
best_psnr = 0.0
history = []

print(f"\n{'='*60}")
print(f"THERMAL 4× SUPER-RESOLUTION TRAINING")
print(f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"LR patch: {PATCH_SIZE//UPSCALE}×{PATCH_SIZE//UPSCALE} → HR patch: {PATCH_SIZE}×{PATCH_SIZE}")
print(f"Checkpoints → {SAVE_DIR}")
print(f"Logs → {log_file}")
print(f"{'='*60}\n")

# ============ TRAINING LOOP ============
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # --- Train ---
    model.train()
    train_loss_sum = 0.0
    train_count = 0

    for step, batch in enumerate(train_loader):
        lr_input = batch['input'].to(device, non_blocking=True)
        hr_target = batch['target'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            hr_pred = model(lr_input)
            loss = criterion(hr_pred, hr_target)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss_sum += loss.item() * lr_input.size(0)
        train_count += lr_input.size(0)

        # Print every 50 steps
        if (step + 1) % 50 == 0:
            print(f"  [Ep {epoch:03d}] Step {step+1}/{len(train_loader)} | "
                  f"Loss: {train_loss_sum/train_count:.6f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    train_loss = train_loss_sum / train_count

    # --- Validate ---
    model.eval()
    val_loss_sum, val_psnr_sum, val_ssim_sum, val_count = 0, 0, 0, 0

    with torch.no_grad():
        for batch in val_loader:
            lr_input = batch['input'].to(device, non_blocking=True)
            hr_target = batch['target'].to(device, non_blocking=True)

            with autocast():
                hr_pred = model(lr_input)
                loss = criterion(hr_pred, hr_target)

            hr_pred = hr_pred.clamp(0, 1)
            val_loss_sum += loss.item()
            val_psnr_sum += calc_psnr(hr_pred, hr_target)
            val_ssim_sum += calc_ssim(hr_pred, hr_target)
            val_count += 1

    val_loss = val_loss_sum / max(1, val_count)
    val_psnr = val_psnr_sum / max(1, val_count)
    val_ssim = val_ssim_sum / max(1, val_count)
    epoch_time = time.time() - epoch_start

    # --- Log ---
    entry = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
             'val_psnr': val_psnr, 'val_ssim': val_ssim,
             'lr': optimizer.param_groups[0]['lr'], 'time': round(epoch_time, 1)}
    history.append(entry)
    with open(log_file, 'a') as f:
        f.write(json.dumps(entry) + '\n')

    # --- Print ---
    print(f"Epoch {epoch:03d}/{EPOCHS} | Train: {train_loss:.6f} | "
          f"Val: {val_loss:.6f} | PSNR: {val_psnr:.2f} | "
          f"SSIM: {val_ssim:.4f} | {epoch_time:.1f}s")

    # --- Save best ---
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        save_path = f"{SAVE_DIR}/best_model.pth"
        torch.save({
            'epoch': epoch, 'best_psnr': best_psnr,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, save_path)
        print(f"  ★ New best PSNR: {best_psnr:.2f} dB → saved!")

    # --- Save periodic ---
    if epoch % 25 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()},
                   f"{SAVE_DIR}/epoch_{epoch:03d}.pth")

print(f"\n{'='*60}")
print(f"✅ Training complete! Best PSNR: {best_psnr:.2f} dB")
print(f"   Model: {SAVE_DIR}/best_model.pth")
print(f"   Log:   {log_file}")
print(f"{'='*60}")

## 6. Visualize Training Curves 📊

In [ ]:
import matplotlib.pyplot as plt

# Load log (works even during training — just re-run this cell)
if not history:
    with open(log_file) as f:
        history = [json.loads(line) for line in f]

epochs_list = [e['epoch'] for e in history]
train_losses = [e['train_loss'] for e in history]
val_losses = [e['val_loss'] for e in history]
val_psnrs = [e['val_psnr'] for e in history]
val_ssims = [e['val_ssim'] for e in history]
lrs = [e['lr'] for e in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('NAFNet-SR Training Progress', fontsize=14, fontweight='bold')

axes[0,0].plot(epochs_list, train_losses, label='Train', color='#2196F3', lw=1.5)
axes[0,0].plot(epochs_list, val_losses, label='Val', color='#F44336', lw=1.5)
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].set_yscale('log')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(epochs_list, val_psnrs, color='#4CAF50', lw=1.5)
best_idx = val_psnrs.index(max(val_psnrs))
axes[0,1].axhline(y=max(val_psnrs), color='#4CAF50', ls='--', alpha=0.5)
axes[0,1].set_title(f'Val PSNR (best: {max(val_psnrs):.2f} dB @ ep {epochs_list[best_idx]})')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(epochs_list, val_ssims, color='#FF9800', lw=1.5)
axes[1,0].set_title('Val SSIM'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(epochs_list, lrs, color='#9C27B0', lw=1.5)
axes[1,1].set_title('Learning Rate'); axes[1,1].set_yscale('log')
axes[1,1].grid(True, alpha=0.3)

for ax in axes.flat: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/training_curves.png', dpi=150)
plt.show()
print(f"Saved to {LOG_DIR}/training_curves.png")

## 7. Inference — Test Your Model 🔍

In [ ]:
#@title Run inference on test images

TEST_LR_DIR = '/content/drive/MyDrive/your_dataset/test/lr'  #@param {type:"string"}
TEST_HR_DIR = '/content/drive/MyDrive/your_dataset/test/hr'  #@param {type:"string"}
RESULTS_DIR = f'{DRIVE_PROJECT}/results'

# Load best model
ckpt = torch.load(f'{SAVE_DIR}/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model (epoch {ckpt['epoch']}, PSNR {ckpt['best_psnr']:.2f})")

os.makedirs(RESULTS_DIR, exist_ok=True)
test_lr_paths = sorted([p for p in Path(TEST_LR_DIR).iterdir() if p.suffix.lower() in IMG_EXT])

# Optional: load HR targets for metrics
has_hr = os.path.exists(TEST_HR_DIR)
psnr_list, ssim_list = [], []

print(f"Processing {len(test_lr_paths)} images...\n")

for i, lr_path in enumerate(test_lr_paths):
    lr_img = load_image(lr_path, 3)
    if lr_img.ndim == 2: lr_img = lr_img[:,:,np.newaxis]
    lr_t = torch.from_numpy(lr_img.transpose(2,0,1)).float().unsqueeze(0).to(device)

    # Pad to multiple of 4
    _, _, h, w = lr_t.shape
    pad_h = (4 - h % 4) % 4
    pad_w = (4 - w % 4) % 4
    if pad_h or pad_w:
        lr_t = F.pad(lr_t, (0, pad_w, 0, pad_h), mode='reflect')

    with torch.no_grad(), autocast():
        hr_pred = model(lr_t)

    # Crop padding
    hr_pred = hr_pred[:, :, :h*UPSCALE, :w*UPSCALE]
    hr_pred = hr_pred.squeeze(0).clamp(0,1).cpu().numpy().transpose(1,2,0)
    hr_pred = (hr_pred * 255).round().astype(np.uint8)

    save_path = f"{RESULTS_DIR}/{lr_path.name}"
    Image.fromarray(hr_pred).save(save_path)

    status = f"[{i+1}/{len(test_lr_paths)}] {lr_path.name} → {save_path}"

    if has_hr:
        hr_path = Path(TEST_HR_DIR) / lr_path.name
        if hr_path.exists():
            hr_gt = load_image(hr_path, 3)
            if hr_gt.ndim == 2: hr_gt = hr_gt[:,:,np.newaxis]
            pred_t = torch.from_numpy(hr_pred.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0)
            gt_t = torch.from_numpy(hr_gt.transpose(2,0,1)).float().unsqueeze(0)
            p = calc_psnr(pred_t, gt_t)
            s = calc_ssim(pred_t, gt_t)
            psnr_list.append(p); ssim_list.append(s)
            status += f" | PSNR: {p:.2f} | SSIM: {s:.4f}"

    print(status)

if psnr_list:
    print(f"\nAvg PSNR: {np.mean(psnr_list):.2f} dB | Avg SSIM: {np.mean(ssim_list):.4f}")
print(f"\n✅ Results saved to {RESULTS_DIR}")

## 8. Visual Comparison 👀
Side-by-side: LR → SR → HR ground truth

In [ ]:
# Show a few random results
n_show = min(4, len(test_lr_paths))
indices = random.sample(range(len(test_lr_paths)), n_show)

fig, axes = plt.subplots(n_show, 3, figsize=(15, 5 * n_show))
if n_show == 1: axes = axes[np.newaxis, :]

for row, idx in enumerate(indices):
    lr_path = test_lr_paths[idx]
    sr_path = f"{RESULTS_DIR}/{lr_path.name}"

    lr_img = np.array(Image.open(lr_path).convert('RGB'))
    sr_img = np.array(Image.open(sr_path).convert('RGB'))

    axes[row, 0].imshow(lr_img)
    axes[row, 0].set_title(f'LR ({lr_img.shape[1]}×{lr_img.shape[0]})', fontsize=12)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(sr_img)
    axes[row, 1].set_title(f'SR Output ({sr_img.shape[1]}×{sr_img.shape[0]})', fontsize=12)
    axes[row, 1].axis('off')

    hr_path = Path(TEST_HR_DIR) / lr_path.name
    if hr_path.exists():
        hr_img = np.array(Image.open(hr_path).convert('RGB'))
        axes[row, 2].imshow(hr_img)
        axes[row, 2].set_title(f'HR Ground Truth ({hr_img.shape[1]}×{hr_img.shape[0]})', fontsize=12)
    else:
        axes[row, 2].text(0.5, 0.5, 'No HR target', ha='center', va='center')
    axes[row, 2].axis('off')

plt.suptitle('LR → Super-Resolved → Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/comparison.png', dpi=150)
plt.show()